# Semaine 5 — Jour 7 : Déploiement d'un système IA

## Position dans la semaine

Ce jour termine la semaine Production AI Systems : architecture, FastAPI, Redis,
PostgreSQL, monitoring, sécurité, puis déploiement. Il suit la roadmap figée du plugin.
La prochaine semaine porte sur les frameworks du marché ; elle n'est pas ajoutée ici.

## Livrable opérationnel

Un lab autonome réunissant une API FastAPI, PostgreSQL et Redis dans Docker Compose.
L'API conserve l'identité et l'isolation par tenant, utilise un cache partitionné,
externalise les quotas dans Redis et expose des métriques Prometheus protégées.
Le fournisseur LLM est simulé ; aucun compte cloud ni clé de fournisseur n'est requis.

## Parcours principal — 60 minutes

| Durée | Activité |
|---|---|
| 5 min | Objectifs, architecture et critères d'acceptation |
| 15 min | Chapitre : image, configuration, sondes, données, rollback |
| 25 min | Installation, démarrage Compose et smoke tests |
| 10 min | Exercices 1 à 3 et comparaison aux corrigés |
| 5 min | Questions d'entretien et checklist de livraison |

Le téléchargement initial des images et dépendances peut allonger le temps de lab.
Les exercices 4–5 et le challenge sont des approfondissements de 30 à 60 minutes.
Sans Docker, exécuter les tests Python ; ils ne remplacent pas le démarrage de la stack.

## Ordre de travail

Lire `learning_objectives.md`, `chapter.md`, puis `labs/README.md`. Le cours explique
les compromis, le README du lab contient les commandes. Les exercices, entretiens
et challenge ont leurs corrigés dans `corriges/`. Deux notebooks sont fournis dans
`notebooks/week05/` à la racine, avec génération depuis Markdown.

## Intégration au dépôt

Ajouter uniquement le jour 7 et ses notebooks. Le lab n'écrase pas les modules des
jours précédents ni les fichiers `__init__.py` partagés. Il reprend les principes
J2–J6 dans un service indépendant afin de pouvoir être lancé sans réunir les anciens ZIP.
Le fichier auth local dérive du jour 6 et ajoute un scope d'exploitation `metrics:read`.

Fusionner les entrées des fichiers de suivi racine avec l'historique existant.
La roadmap fournie est inchangée. Cette archive est un incrément, pas une copie
intégrale du dépôt.

## Critères de réussite

- API exécutée par un utilisateur non root ; données hors image.
- DB et Redis sans ports publiés ; API limitée à localhost pour le lab.
- Secrets absents de l'image et montés uniquement dans les services nécessaires.
- Readiness contrôlant schéma PostgreSQL et disponibilité Redis.
- Lecture alpha autorisée, beta refusée, cache sans contournement d'ACL.
- Quota atomique partagé et refus contrôlé si Redis est indisponible.
- Smoke tests, procédure de release et rollback compris.

Voir `assets/validation_report.md` pour distinguer les contrôles exécutés des
vérifications Docker restant à effectuer sur votre machine.


# Objectifs — Déploiement IA

À la fin du jour 7, savoir :

1. Distinguer image, conteneur, configuration et données persistantes.
2. Construire une image applicative sans secrets et avec un UID non root.
3. Exécuter une stack locale avec réseaux, volumes et secrets explicites.
4. Distinguer liveness, readiness, disponibilité et validation métier.
5. Expliquer le rôle et les limites de `depends_on: service_healthy`.
6. Tester un cache et un quota partagés sans mélanger les tenants.
7. Livrer une version identifiée et vérifier son fonctionnement après démarrage.
8. Revenir à l'image précédente sans prétendre restaurer automatiquement la base.
9. Identifier les adaptations nécessaires avant un déploiement public.

## Preuves attendues

Une configuration Compose comprise, les tests Python exécutés, les neuf contrôles
de `smoke_test.py` réussis sur votre machine, et une explication du rollback avec
conservation des volumes. La validation pédagogique reste à réaliser par l'apprenant.

## Portée

Un hôte local ou une VM unique suffit au lab. La haute disponibilité, le provisioning
OpenTofu/Ansible, l'installation d'un cluster et la publication d'un service public
ne sont pas réalisés par cette livraison. Ils sont discutés comme choix d'architecture.


# Chapitre — Livrer un service IA exploitable

## 1. Déployer n'est pas seulement lancer un processus

Aux jours précédents, les composants étaient testables séparément. Maintenant il
faut réunir code, configuration, données et politique de sécurité dans une exécution
contrôlée. Un service qui démarre peut encore être incapable de servir : mauvais
schéma SQL, mot de passe périmé, Redis inaccessible, cache mal partitionné.

Le déploiement doit répondre à quatre questions : quelle version tourne, avec
quelle configuration, sur quelles données et comment revenir à un état utilisable ?

Le lab choisit Docker Compose sur un seul hôte. C'est un périmètre adapté à une
première stack reproductible. Il ne constitue pas une architecture à haute disponibilité.
Vous connaissez déjà Docker et les pipelines : l'objectif est d'appliquer leurs
mécanismes aux particularités d'un backend IA.

## 2. Image, conteneur, configuration, données

| Élément | Contenu | Cycle de vie |
|---|---|---|
| Image API | Code, interpréteur et dépendances | Construite une fois par version |
| Conteneur | Processus API et filesystem d'exécution | Remplaçable |
| Configuration | Hôtes, nom de base, référence de release | Fournie au démarrage |
| Secrets | Clés client et mots de passe | Distribués avec accès limité |
| Volumes | Données PostgreSQL, état Redis | Conservés entre remplacements |

Reconstruire l'image ne met pas automatiquement à jour la base. Redémarrer un
conteneur ne réinitialise pas ses volumes. Remplacer un secret PostgreSQL dans un
fichier ne change pas le mot de passe déjà enregistré dans une base existante.
Ces distinctions expliquent une grande partie des incidents de premier déploiement.

## 3. Architecture du lab

```mermaid
flowchart TD
    C[Client localhost] --> A[API FastAPI]
    A --> P[(PostgreSQL)]
    A --> R[(Redis)]
    S[Secrets montés] --> A
    S --> P
    S --> R
    A --> M[Métriques et audit]
```

Seule l'API publie le port 8000, lié à `127.0.0.1`. PostgreSQL et Redis restent sur
un réseau backend interne. L'API est également connectée à un réseau frontend,
ce qui permettra ultérieurement une connexion à un vrai fournisseur LLM.

Le lab ne fournit pas de reverse proxy TLS. Il n'est donc pas configuré pour être
exposé directement sur Internet. Un déploiement public ajoute TLS, contrôle d'entrée,
limite de corps HTTP, délais et limitation avant authentification.

Le réseau interne réduit l'exposition ; il ne remplace ni authentification ni
moindre privilège. Redis exige un mot de passe. PostgreSQL possède un compte
applicatif de lecture, distinct du compte administrateur de bootstrap.

## 4. Construire l'image avec une responsabilité précise

Le Dockerfile utilise une image Python slim, installe les dépendances runtime et
copie uniquement le package applicatif. Il crée l'UID/GID 10001 puis exécute Uvicorn
par un CMD au format exec. Aucun mode `--reload` n'est activé.

`.dockerignore` applique une liste d'inclusion : seuls le code et les fichiers de
dépendances nécessaires entrent dans le contexte. Les secrets, tests et environnements
virtuels ne sont pas envoyés au moteur de build.

L'UID 10001 n'est pas un utilisateur IAM ni un compte client. C'est l'identité Linux
du processus dans le conteneur. Le filesystem API est en lecture seule ; `/tmp` est
un tmpfs borné. Les capacités Linux sont retirées et l'élévation de privilèges interdite.

Le fichier `requirements.lock` contient les versions runtime résolues pour Linux et
Python 3.12. Il évite un choix implicite différent des dépendances transitives lors
d'un build. Il n'inclut pas les dépendances de test. Ce n'est pas un lock multi-plateforme
avec hashes : ce niveau supplémentaire reste une amélioration d'industrialisation.

Les images de base Python, PostgreSQL et Redis utilisent ici des tags de gamme pour
la facilité du lab. Ces tags sont mutables. Pour une livraison durable, enregistrer
et tester leurs digests, le commit de source, l'architecture et les résultats de scan.
Un tag `release-001` lisible ne prouve pas à lui seul l'immutabilité ; une référence
`image@sha256:...` la rend explicite pour une image publiée dans un registry.

## 5. Secrets : montage et moindre privilège

Le script `configure_demo.py` crée des valeurs aléatoires sans les afficher. Chaque
service reçoit seulement les secrets nécessaires : l'API ne reçoit jamais le mot de
passe administrateur PostgreSQL.

Les secrets sont montés en fichiers sous `/run/secrets`. L'environnement ne contient
que leurs chemins et la configuration non sensible. Le code les lit au démarrage et
refuse une configuration absente ou invalide.

Avec Compose et des fichiers locaux, il ne s'agit pas d'un coffre-fort chiffré.
Le répertoire hôte `secrets/` est privé en mode 0700 sur Unix. Les fichiers sont
lisibles en mode 0444 pour que les UID des services puissent les lire après montage.
Cette combinaison est volontaire : les implémentations de secrets basées sur des
bind mounts ne garantissent pas un remappage d'UID/mode par la déclaration Compose.
Sur un autre OS, vérifier les ACL et le comportement du moteur Docker.

Une rotation doit coordonner le fichier, l'état du service cible et ses consommateurs.
Le script refuse de régénérer un dossier existant pour éviter de désynchroniser
les mots de passe d'une base déjà initialisée. La suppression du dossier de secrets
n'est pas une méthode de rotation.

## 6. Ordre de démarrage et santé

Compose peut attendre des dépendances déclarées `service_healthy` avant de créer
l'API. C'est utile pour la première ouverture de PostgreSQL et Redis. Cette attente
ne garantit pas qu'ils resteront disponibles après le démarrage.

`pg_isready` constate qu'un serveur PostgreSQL accepte les connexions ; il ne prouve
pas que le compte applicatif lit le bon schéma. La readiness API vérifie donc la
table `schema_version` via le compte applicatif et attend la version 1.

| Contrôle | Question | Comportement du lab |
|---|---|---|
| `/health/live` | Le processus peut-il répondre ? | Ne consulte pas les dépendances |
| `/health/ready` | Le service peut-il traiter son contrat ? | Consulte schéma SQL et Redis |
| Healthcheck Compose API | Le conteneur passe-t-il le contrôle de livraison ? | Utilise readiness |
| Smoke tests | Les fonctions importantes sont-elles correctes ? | Auth, données, cache, métriques |

Un problème PostgreSQL donne une readiness 503 sans faire tomber la liveness.
Redémarrer continuellement une API parce qu'une base est momentanément indisponible
peut aggraver l'incident. Docker ne redémarre pas automatiquement un conteneur
simplement parce qu'il est marqué unhealthy ; sa politique de restart concerne
l'arrêt du processus.

Les probes comportent des timeouts. La connexion PostgreSQL est bornée à trois
secondes et ses requêtes à deux secondes ; Redis est borné à deux secondes. La
readiness consulte les deux séquentiellement et reste sous le timeout du healthcheck.
Des délais réseau différents doivent être validés dans l'environnement cible.

## 7. Données et schéma : initialiser n'est pas migrer

Le bootstrap PostgreSQL s'exécute à l'initialisation d'un volume vide. Il crée un
compte de lecture, le schéma minimal et les documents alpha/beta.

Les scripts de `/docker-entrypoint-initdb.d` ne sont pas rejoués sur un volume
existant. Modifier le script puis relancer Compose ne constitue donc pas une migration.
Pour une évolution réelle, utiliser un mécanisme de migration versionné, une
identité privilégiée dédiée et une étape de livraison observable.

La requête documentaire applique le tenant du principal dans son WHERE et passe
les arguments au driver. Le contrôle précède le cache. Le compte applicatif ne peut
ni modifier les documents ni créer une table ; l'isolation par tenant reste ici
appliquée par le code SQL, sans Row Level Security.

Le volume PostgreSQL conserve les données lorsque l'API ou la stack est arrêtée
avec `docker compose down`. L'option `--volumes` détruit les volumes nommés de cette
stack : elle est réservée au nettoyage explicite d'un lab jetable ou d'un runner CI.

Un volume n'est pas une sauvegarde. Avant une migration, prévoir sauvegarde et test
de restauration. Revenir à l'image précédente ne remet pas automatiquement la base
dans son ancien schéma.

## 8. Cache et quotas à plusieurs processus

Le cache de réponses inclut tenant, sujet, version du contrat et une empreinte du
message et du contenu documentaire. Le document est relu et autorisé avant tout
cache hit. Une mise à jour de son contenu change l'empreinte ; une révocation empêche
la lecture avant de consulter le cache.

Cette relecture réduit le gain sur l'accès SQL mais conserve le gain éventuel sur
l'appel LLM. C'est un compromis explicite de ce lab. Une architecture plus performante
peut utiliser une version d'ACL et de document, à condition de maîtriser l'invalidation.
Les réponses sont conservées 120 secondes ; leur présence dans Redis reste une
forme de stockage de données sensibles. Le préfixe `answer:v1` doit être changé si
le contrat de génération, le prompt ou le modèle change. `APP_RELEASE` ne l’invalide
pas automatiquement : une nouvelle image peut sinon réutiliser une ancienne réponse.

Le quota passe du verrou local du jour 6 à un script Lua atomique dans Redis. Le
contrôle, l'incrément et l'expiration sont réalisés dans la même invocation. Le
couple tenant/sujet partage le quota entre instances ; changer de clé ne l'annule pas.

Redis porte désormais une frontière de consommation : si le limiteur est inaccessible,
l'API renvoie 503 au lieu d'accepter des appels sans borne. Le cache ne peut donc pas
être considéré ici comme une dépendance entièrement facultative. Distinguer les
politiques de cache et de quota dans une architecture plus avancée.

Redis est configuré `noeviction` pour éviter qu'une pression mémoire supprime un
compteur de quota et accorde un nouveau budget. Une saturation provoque des erreurs
à traiter et à surveiller. AOF `everysec` améliore la persistance mais ne donne pas
une garantie de conservation de chaque incrément en cas de crash. Pour un budget
financier strict, définir des garanties et un rapprochement adaptés.

Une fenêtre fixe limite dix appels sur soixante secondes à partir du premier appel.
Elle autorise une rafale autour de sa frontière ; elle n'est pas un plafond monétaire
ni une protection complète contre des requêtes anonymes.

## 9. Observabilité lors d'une release

Le lab expose des compteurs HTTP, un histogramme de latence et les cache hits au
format Prometheus. `/metrics` demande le scope `metrics:read`, attribué à une identité
d'exploitation distincte. Une clé de chat ne donne pas cet accès.

Les labels sont route et statut. Ni clé, ni sujet, ni request_id ne créent une série
de métrique. L'audit JSON conserve des métadonnées et un identifiant généré par le
serveur sans lire le prompt. Les logs Docker sont bornés en taille et nombre de fichiers.
Ils doivent ensuite être collectés selon la politique de l'environnement.

Le lab lance un seul worker API. Les métriques sont locales à ce processus ; un
passage à plusieurs workers impose une collecte adaptée ou plusieurs conteneurs
scrapés séparément. Le quota partagé via Redis ne résout pas cette question de métriques.

Les tokens et coûts réels ne sont pas mesurés parce que le modèle est simulé.
Ne pas présenter un compteur d'appels comme une mesure de facturation.

## 10. Pipeline : build une fois, promouvoir le résultat

Une chaîne de livraison vérifie le code et les contrats, scanne les dépendances et
l'image, construit une image identifiée, puis teste cette image dans un environnement
jetable. La promotion utilise ensuite le même digest, plutôt qu'un nouveau build
avec potentiellement d'autres dépendances.

Le fichier `assets/ci_pipeline.yml` est un exemple GitHub Actions pour un dépôt public.
Il utilise des commandes Git et shell, sans action externe, pour respecter les
contraintes d'allowlist déjà rencontrées dans le projet. Il teste le code, construit
l'image et exécute une stack jetable avec les smoke tests.

Il ne pousse pas d'image et ne déploie pas une VM. Adapter l'authentification Git
pour un dépôt privé. Ajouter vos étapes de scan Trivy/SBOM et votre politique de
refus avant une promotion réelle ; elles ne sont pas revendiquées comme exécutées ici.

## 11. Mettre à jour et revenir en arrière

Pour ce lab, conserver l'image `release-001`, construire une `release-002` après
un changement et appliquer uniquement le service API. Les volumes restent en place.
Attendre sa readiness, exécuter les smoke tests puis observer les erreurs et la latence.

Si ces contrôles échouent, restaurer la référence précédente et vérifier à nouveau
readiness et smoke tests. Ne pas conclure que le rollback a réussi simplement parce
que la commande de redémarrage a terminé.

Cette procédure a une interruption possible pendant le remplacement. Compose ne
fournit pas automatiquement un déploiement blue/green ou une bascule sans coupure.
Avec un proxy, deux instances peuvent être validées avant la bascule ; leurs schémas,
quotas et contrats doivent alors être compatibles simultanément.

Une migration destructive peut rendre l'ancienne image inutilisable. Une stratégie
expand/contract conserve d'abord la compatibilité, migre les consommateurs, puis
retire les anciens éléments dans une release ultérieure.

Le challenge fournit un contrôleur de décisions avec callbacks simulés. Il vérifie
promotion et rollback, sans prétendre piloter un vrai moteur Docker.

## 12. Arrêt et reprise

Le CMD exec, `init: true` et la période d'arrêt permettent à Uvicorn de recevoir les
signaux et de terminer les requêtes dans sa fenêtre de shutdown. Le timeout graceful
est inférieur à la période de grâce du conteneur. Cela réduit les coupures mais ne
garantit pas qu'un appel fournisseur interrompu n'a pas été facturé.

Pour des travaux longs, envisager une file durable, un état persistant et un protocole
d'idempotence. Redémarrer un processus ne doit pas déclencher une deuxième action
irréversible. Le lab ne contient aucune écriture métier ni worker asynchrone.

## 13. Choisir le support de production

| Support | Adapté à | Limite principale |
|---|---|---|
| Compose sur VM | Petite stack, apprentissage, service mono-hôte | Dépendance à un seul hôte et opérations à organiser |
| Service de conteneurs géré | Déploiement standard avec exploitation allégée | Intégration identité/réseau et garanties propres au service |
| Kubernetes | Plusieurs services, politiques et orchestration à grande échelle | Complexité opérationnelle et prérequis d'exploitation |

OpenTofu/Terraform crée les ressources cloud ; Ansible peut préparer l'hôte ; Compose
ou l'orchestrateur exécute les conteneurs. Le compte cloud du provisioner, l'utilisateur
SSH et l'UID du conteneur ont des responsabilités distinctes. Un utilisateur IAM n'est
pas automatiquement un utilisateur Linux ou un principal de votre API.

Le critère de choix est le besoin de disponibilité et d'exploitation, pas la présence
d'IA. Avant une exposition publique : authentification adaptée, TLS, tailles et délais,
secrets gérés, stockage sauvegardé, tests de restauration, quotas, egress, monitoring
et procédure d'incident doivent être validés ensemble.

## 14. Ce que vous devez pouvoir expliquer

Quelle image exécute le code ? Où vivent les données ? Quels secrets lit chaque
service ? Que signifie healthy ? Que se passe-t-il si Redis tombe ? Quelle image
restaurer et avec quel schéma ? Ces réponses constituent le contrat de livraison
que l'équipe d'exploitation doit pouvoir vérifier.


# Atelier exécutable — Déploiement sans moteur Docker

Les cellules utilisent les dépendances de `labs/requirements-dev.txt`. Elles
exécutent les contrats API et le Lua Redis en mémoire, sans preuve de déploiement
ou de connexion à PostgreSQL réel. Les commandes de la stack sont dans `labs/README.md`.

## 1. Retrouver le lab et préparer des dépendances de test

In [ ]:
from pathlib import Path
import sys
import secrets

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "book/week05/day07/labs").is_dir()), None)
if root is None:
    raise RuntimeError("Ouvrir le notebook depuis ai-engineering-bootcamp ou ses sous-dossiers.")
lab = root / "book/week05/day07/labs"
if str(lab) not in sys.path:
    sys.path.insert(0, str(lab))

import fakeredis
from fastapi.testclient import TestClient
from deployment_ai_service.app import create_app
from deployment_ai_service.auth import Credential, IdentityStore, Principal
from deployment_ai_service.storage import Document, DocumentNotFound, RedisBackend, RateLimited, require_scope

class DemoRepository:
    def __init__(self):
        self.available = True
    def ready(self):
        return self.available
    def get(self, principal, document_id):
        require_scope(principal, "documents:read")
        if principal.tenant != "alpha" or document_id != "alpha-guide":
            raise DocumentNotFound
        return Document("alpha-guide", "Guide privé alpha")

alice = Principal("alice", "alpha", frozenset({"chat:write", "documents:read"}))
operator = Principal("operator", "platform", frozenset({"metrics:read"}))
key, ops_key = secrets.token_urlsafe(32), secrets.token_urlsafe(32)
identities = IdentityStore([Credential(key, alice), Credential(ops_key, operator)])
repository = DemoRepository()
redis_client = fakeredis.FakeRedis(decode_responses=True)
backend = RedisBackend(redis_client, limit=100)
client = TestClient(create_app(identities, repository, backend))
headers, ops_headers = {"X-API-Key": key}, {"X-API-Key": ops_key}
print("Dépendances de test prêtes ; aucun secret affiché")

## 2. Distinguer santé du processus et readiness

In [ ]:
assert client.get("/health/live").status_code == 200
assert client.get("/health/ready").status_code == 200
repository.available = False
assert client.get("/health/live").status_code == 200
assert client.get("/health/ready").status_code == 503
repository.available = True
print("Liveness indépendante de la disponibilité SQL")

## 3. Vérifier cache, isolation et métriques

In [ ]:
payload = {"message": "Résume", "document_id": "alpha-guide"}
first = client.post("/chat", json=payload, headers=headers)
second = client.post("/chat", json=payload, headers=headers)
assert first.status_code == second.status_code == 200
assert first.json()["cached"] is False
assert second.json()["cached"] is True
assert client.get("/documents/beta-guide", headers=headers).status_code == 404
assert client.get("/metrics", headers=headers).status_code == 403
metrics = client.get("/metrics", headers=ops_headers)
assert metrics.status_code == 200 and "ai_cache_hits_total" in metrics.text
assert key not in metrics.text
print("Cache autorisé, beta refusé et métriques protégées")

## 4. Démontrer un quota commun à deux instances

In [ ]:
shared = fakeredis.FakeRedis(decode_responses=True)
instance_one = RedisBackend(shared, limit=1)
instance_two = RedisBackend(shared, limit=1)
instance_one.consume(alice)
try:
    instance_two.consume(alice)
except RateLimited as error:
    assert 1 <= error.retry_after <= 60
    print("Deuxième instance limitée par le même quota")
else:
    raise AssertionError("Le quota n'est pas partagé")
client.close()

## 5. Passage à la stack réelle

Sur une machine disposant de Docker, suivre le README du lab, puis comparer :
sondes, lecture SQL réelle, données conservées après redémarrage, quotas partagés et
smoke tests. Les résultats ci-dessus ne prouvent ni l'image, ni les montages, ni
le bootstrap PostgreSQL. Résoudre les exercices avant de consulter leurs corrigés.

# Exercices — Déploiement

## Exercice 1 — Healthy, ready ou correct ? (principal)

L'API retourne 200 sur `/health/live`, PostgreSQL répond à `pg_isready`, mais la
readiness retourne 503. Proposer deux causes compatibles avec ces observations et
la vérification utile pour chacune. Pourquoi la liveness ne consulte-t-elle pas la DB ?

## Exercice 2 — Données et mots de passe (principal)

Après un premier démarrage, vous modifiez le fichier du mot de passe applicatif et
le script SQL d'initialisation, puis relancez Compose. L'API ne se connecte plus.
Expliquer pourquoi. Décrire une rotation et une migration correctes sans supprimer
le volume pour « réparer ».

## Exercice 3 — Choisir un rollback (principal)

Release-002 utilise une nouvelle colonne obligatoire et la migration a supprimé
une ancienne colonne requise par release-001. Les smoke tests échouent.
Peut-on simplement restaurer l'image 001 ? Proposer une séquence expand/contract.

## Exercice 4 — Partage du quota (approfondissement)

Instancier deux backends pointant vers la même instance fakeredis, puis faire consommer
le quota du même principal. Vérifier que changer de backend ne crée pas un nouveau
budget. Expliquer la différence entre ce test et deux Redis indépendants.

## Exercice 5 — Contrôle de livraison (approfondissement)

Compléter une checklist avant/après livraison : référence d'image, configuration,
schéma, sauvegarde, readiness, tests fonctionnels, isolation, métriques et rollback.
Pour chaque élément, préciser une preuve concrète plutôt qu'un simple « OK ».


# Questions d'entretien — Déploiement IA

1. Que doit contenir l'image d'un service IA et que faut-il fournir au démarrage ?
2. Pourquoi `depends_on` ne remplace-t-il pas une stratégie de reprise réseau ?
3. Quelle différence entre liveness et readiness ?
4. Pourquoi un UID 10001 n'est-il pas un utilisateur IAM ou une identité client ?
5. Une réponse en cache peut-elle contourner une révocation documentaire ?
6. Comment rendre un quota correct lorsque plusieurs instances servent la même identité ?
7. Pourquoi une release avec un tag unique peut-elle encore manquer de reproductibilité ?
8. Que conserve `docker compose down` et que détruit `--volumes` ?
9. Quelles conditions permettent un rollback d'image après migration ?
10. Quels risques particuliers apportent les appels LLM longs et les retries ?

## Mise en situation

Une release passe la readiness mais produit des 403 pour tous les utilisateurs et
aucun cache hit. Proposer le diagnostic et les critères pour conserver ou annuler
la release. Ne pas journaliser les clés client pour faciliter la recherche.


# Challenge — Décider une promotion ou un rollback

## Contrat

Écrire un contrôleur recevant : référence précédente, référence candidate,
`deploy(release)`, `readiness()` et `smoke()`. Les callbacks sont simulés.

- Refuser deux références identiques ou absentes.
- Refuser avant toute mutation un schéma déclaré incompatible avec le rollback.
- Déployer la candidate et vérifier readiness puis smoke tests.
- Si le déploiement ou un contrôle échoue, restaurer la précédente et revérifier.
- Si la récupération échoue, signaler explicitement une intervention nécessaire.
- Retourner la version active et si un rollback a eu lieu.

Chaque callback doit avoir un délai maximal. Le corrigé montre la décision mais
n'implémente ni le délai interne du moteur Docker ni une bascule réseau sans interruption.
L'indicateur de compatibilité vient de votre processus de livraison : le contrôleur
ne peut pas déduire la compatibilité de deux schémas à partir d'un booléen.

## Preuves attendues

Tests de succès, readiness KO, smoke KO, exception de déploiement, schéma incompatible
et rollback KO. Une note explique comment relier les callbacks aux commandes du lab
sans supprimer les volumes. Temps conseillé : 30 à 45 minutes.


# Corrigés des exercices

## 1. Santé et readiness

PostgreSQL peut accepter des connexions mais refuser le compte applicatif après une
rotation incomplète ; tester ce compte sans imprimer son secret. Le serveur peut
également fonctionner avec un schéma absent ou d'une version différente ; vérifier
le bootstrap ou la migration et la table de version.

Redis peut être indisponible alors que PostgreSQL fonctionne. Examiner son healthcheck
et sa configuration, sans exposer les mots de passe. La liveness doit constater le
processus, pour ne pas créer une boucle de redémarrage liée à une dépendance externe.

## 2. Mot de passe et bootstrap

Les scripts d'initialisation ne sont exécutés que sur un volume vide. Le rôle SQL
conserve donc l'ancien mot de passe, tandis que l'API lit le nouveau fichier. Modifier
le script n'applique pas non plus une évolution sur une base existante.

Planifier la rotation : mettre à jour le rôle via un accès privilégié protégé,
mettre à jour le secret distribué aux consommateurs, redémarrer/recharger ceux-ci,
puis vérifier une connexion et la readiness. Organiser le chevauchement si requis
pour éviter l'interruption. Le compte applicatif ne peut pas réaliser lui-même cette
administration. Pour le schéma, utiliser une migration versionnée ; conserver et
tester une sauvegarde. Ne pas supprimer des données pour contourner la procédure.

## 3. Rollback et schéma

Non : l'ancienne image dépend d'une colonne supprimée. Un rollback d'image seul
échoue ou réintroduit des erreurs. Corriger vers l'avant ou restaurer selon un plan
explicite avec conséquences sur les données.

Expand/contract : ajouter la nouvelle colonne de façon compatible ; permettre aux
deux versions de fonctionner ; migrer/backfiller ; déployer les lecteurs/écrivains
nouveaux ; vérifier qu'aucun ancien consommateur ne reste ; supprimer l'ancienne
colonne dans une release ultérieure. La phase destructive ferme la fenêtre de
rollback simple et doit être traitée comme telle.

## 4. Partage du quota

```python
import fakeredis
from deployment_ai_service.auth import Principal
from deployment_ai_service.storage import RedisBackend, RateLimited

client = fakeredis.FakeRedis(decode_responses=True)
alice = Principal("alice", "alpha", frozenset({"chat:write"}))
first, second = RedisBackend(client, limit=1), RedisBackend(client, limit=1)
first.consume(alice)
try:
    second.consume(alice)
except RateLimited:
    print("Quota partagé : deuxième appel refusé")
else:
    raise AssertionError("Quota contourné")
```

Deux clients Redis connectés à des serveurs distincts ne partageraient pas le même
état. Le test fakeredis exécute la logique Lua sans prouver la configuration réseau
et la persistance de Redis dans le conteneur.

## 5. Checklist de livraison

| Élément | Preuve |
|---|---|
| Image | Commit associé, digest/identifiant conservé, résultat du scan |
| Configuration | Valeurs non sensibles relues et chemins de secrets validés |
| Schéma | Version compatible lue avec le compte applicatif |
| Sauvegarde | Identifiant du backup et résultat du test de restauration |
| Readiness | 200 observé après remplacement et lors de la montée en charge |
| Métier | Smoke tests auth, document, cache et métriques réussis |
| Isolation | Même ressource beta refusée sous identité alpha |
| Exploitation | Erreurs, latence et logs surveillés sur une fenêtre définie |
| Rollback | Image précédente disponible, compatibilité et procédure de retour testées |

Un bon diagnostic exige des preuves complémentaires : une seule sonde ou un seul
smoke test ne couvre pas tous les comportements.


# Corrigé de l'entretien

1. Image : code et runtime versionnés. Démarrage : configuration, secrets et accès
   aux données. Les données et clés client ne doivent pas être intégrées à l'image.
2. L'ordre initial ne protège pas contre une panne ultérieure. Timeouts, réponses
   contrôlées, reconnexion et politique de retry doivent vivre dans l'application.
3. Liveness constate le processus ; readiness sa capacité à servir le contrat. Ne
   pas redémarrer l'API en boucle pour une indisponibilité SQL externe.
4. L'UID est une identité Linux du processus. IAM contrôle le cloud ; le principal
   API représente un client. Ces autorités ne s'accordent pas automatiquement.
5. Oui si l'ACL n'est pas revalidée. Le lab relit le document avant tout cache hit et
   partitionne la clé par tenant et sujet. La révocation reste effective.
6. Utiliser un état partagé et une mutation atomique. Redis Lua évite l'intercalation
   entre contrôle et incrément. Définir la conduite lors d'une panne du limiteur.
7. Le tag peut être déplacé ; la base de l'image et les dépendances transitives
   peuvent varier. Conserver digest, lock, plateforme et provenance du build.
8. `down` retire les conteneurs/réseaux gérés tout en conservant les volumes nommés.
   `--volumes` retire ceux de la stack ; ce n'est pas une étape habituelle de rollback.
9. Compatibilité du schéma et des données avec l'ancienne image, disponibilité de
   cette image et revalidation. Expand/contract maintient une fenêtre de retour.
10. Un timeout ne prouve pas l'absence de consommation. Borner sorties et retries,
    réserver les budgets, rapprocher les appels et gérer l'idempotence des actions.

## Mise en situation

La readiness prouve la disponibilité des dépendances, pas l'attribution correcte
des scopes. Examiner les métadonnées des identités et la politique de route, puis
les changements de schéma de cache et de contexte d'autorisation. Comparer au
comportement précédent avec un compte de test autorisé.

Conserver la release seulement si les écarts sont attendus et acceptés. Si les
utilisateurs légitimes ont perdu l'accès, restaurer la version/configuration
compatible précédente et vérifier les scénarios métier. Ne pas diminuer les
permissions pour rendre tous les appels 200 ni exposer les clés dans les logs.


# Corrigé — Contrôler promotion et rollback

## Décision

La readiness prouve la capacité à servir les dépendances, puis les smoke tests
vérifient les chemins fonctionnels importants. La récupération doit être vérifiée
avec les mêmes contrôles. Une compatibilité de schéma inconnue bloque la livraison.

## Implémentation exécutable

Ce code correspond à `labs/rollout_solution.py`. Les callbacks ne lancent aucun
conteneur ; ils simulent les décisions que la procédure manuelle doit prendre.

In [ ]:
"""Corrigé de décision : callbacks simulés, aucune commande Docker exécutée ici."""
from dataclasses import dataclass


class RolloutFailed(Exception):
    pass


@dataclass(frozen=True)
class RolloutResult:
    active_release: str
    rolled_back: bool


def rollout(previous, candidate, deploy, readiness, smoke, schema_compatible=True):
    if not previous or not candidate or previous == candidate:
        raise ValueError("Distinct previous and candidate releases are required")
    if not schema_compatible:
        raise ValueError("Schema is incompatible with rollback")
    try:
        # Le callback deploy doit contenir son propre délai maximal.
        deploy(candidate)
        if not readiness() or not smoke():
            raise RolloutFailed("Candidate failed release checks")
        return RolloutResult(candidate, False)
    except Exception:
        try:
            deploy(previous)
            if not readiness() or not smoke():
                raise RolloutFailed("Previous release failed recovery checks")
        except Exception:
            raise RolloutFailed("Rollback failed; manual recovery required") from None
        return RolloutResult(previous, True)

## Démonstration d'un échec de candidate

In [ ]:
deployed = []
smoke_results = iter([False, True])
result = rollout(
    "release-001", "release-002", deployed.append,
    lambda: True, lambda: next(smoke_results), schema_compatible=True,
)
assert deployed == ["release-002", "release-001"]
assert result.active_release == "release-001" and result.rolled_back
print("Candidate refusée ; précédente revalidée")

## Adapter les callbacks

`deploy` doit exécuter l'application de l'image exacte, attendre avec un délai maximal
et conserver stdout/stderr sans secrets. Un code de retour non nul lève une erreur.
`readiness` effectue des sondes bornées ; `smoke` exécute les scénarios avec une identité
de test. Une attente infinie à l'intérieur d'un callback empêcherait le contrôleur
de lancer sa récupération.

La version précédente doit être disponible avant mutation. Prévoir l'exclusion
mutuelle de deux pipelines sur le même environnement et vérifier le digest actif.
Ne pas utiliser un booléen fabriqué pour contourner une incompatibilité de migration.
La compatibilité relève du contrat de schéma et de la stratégie expand/contract.

## Limites

Le contrôleur ne restaure ni DB ni configuration de secrets, ne journalise pas un
historique de release et ne garantit pas de bascule sans interruption. Une erreur
peut survenir après une mutation partielle ; sa récupération exige l'idempotence
des callbacks. Les tests prouvent le contrat de décision, pas un déploiement Docker.

# Revue pédagogique — Déploiement

## Ce qui doit être maîtrisé

Séparation code/configuration/données ; santé applicative et des dépendances ;
identité non root et autorisation API ; quota partagé ; cache avec contrôle d'accès ;
référence de release et compatibilité de schéma.

## Pièges à repérer

Assimiler une image à une sauvegarde ; croire que l'init SQL rejoue les migrations ;
modifier un fichier de secret sans modifier le service cible ; oublier les clients
sur un ancien secret ; effacer des volumes lors du rollback ; conclure à la réussite
avec une seule liveness ; annoncer du zéro downtime avec le remplacement Compose.

## Barème proposé

| Critère | Points |
|---|---|
| Image et secrets correctement expliqués | 4 |
| Stack, volumes et santé compris | 5 |
| Sécurité, cache et quotas vérifiés | 4 |
| Livraison/rollback justifiés | 5 |
| Limites de validation reconnues | 2 |

Réussite proposée : 14/20 sans suppression involontaire de données ni fuite inter-tenant.
La validation pédagogique appartient à l'apprenant ; les fichiers de suivi ne
confondent pas production de contenu et achèvement de la formation.

## Propositions d'amélioration

Sans changer les spécifications : verrouiller les images par digest, ajouter une
collecte métrique, pratiquer une restauration PostgreSQL et intégrer un vrai
fournisseur à une campagne de tests adversariaux. Le provisioning cloud et une
bascule blue/green peuvent prolonger le lab sans modifier la roadmap.


# Références — Déploiement

## Sources primaires consultées le 2 octobre 2026

- Docker, ordre de démarrage : https://docs.docker.com/compose/how-tos/startup-order/
- Docker, production Compose : https://docs.docker.com/compose/how-tos/production/
- Docker, services : https://docs.docker.com/reference/compose-file/services/
- Docker, secrets : https://docs.docker.com/reference/compose-file/secrets/
- FastAPI, conteneurs : https://fastapi.tiangolo.com/deployment/docker/
- Redis, INCR et script atomique : https://redis.io/docs/latest/commands/incr/
- PostgreSQL 16, psql et getenv : https://www.postgresql.org/docs/16/app-psql.html

## Raccordement interne

J2 : API et validation. J3 : cache et TTL. J4 : requêtes et persistance.
J5 : métriques et diagnostic. J6 : identité, tenant, secrets et limites.
Le J7 reprend ces principes dans une stack déployable autonome.
